# Lexora — Dyslexia Screening Pipeline
**Based on:** Rello et al. (2020) *Predicting risk of dyslexia with an online gamified test*, PLOS ONE

This notebook reproduces and extends the paper's methodology:
- Combines desktop (N=3,644) + tablet (N=1,395) datasets
- Splits participants into **3 age groups** with group-specific question sets
- Handles class imbalance with SMOTE + class weights
- Trains a Random Forest per age group
- Exports `.pkl` model files for use in the Lexora web app

---
### Methodology Note — English/Arabic Adaptation
The original test was designed for Spanish (a transparent orthography). For Lexora, exercises were adapted to English and Arabic following these principles:
1. **Phoneme–grapheme mapping**: Spanish mirror-letter confusions (b/d, p/q, n/u) are universal and were kept as-is.
2. **Phonological awareness tasks**: Replaced Spanish syllable pairs (ba/da, glis/glir) with English equivalents targeting similar phoneme contrasts (e.g. /b/–/d/, /p/–/q/, consonant clusters).
3. **Lexical/spelling tasks**: Spanish words with dyslexic error patterns were replaced with English words of matched frequency and phonological complexity (e.g. 'siete→seite' became 'stone→stoне').
4. **Arabic**: RTL layout applied; Arabic phoneme pairs selected from documented Arabic dyslexia error corpora (ب/ت/ث confusions, ع/غ, و/ر).
5. **Gamification layer**: Identical game mechanics (Whac-A-Mole timing, drag-reorder) preserved so interaction features remain comparable.

---
### ⚠️ Disclaimer
> **This screening tool is NOT a diagnostic instrument.** Results indicate statistical risk only and must never replace a professional evaluation by a licensed speech-language pathologist, educational psychologist, or neuropsychologist. A positive screen means a child *may benefit* from a professional assessment — it does not confirm dyslexia. False positives and false negatives are possible. This tool does not screen for co-occurring conditions (ADHD, dyscalculia, SLI). Lexora follows the ethical framework of the original Rello et al. (2020) study and the IRB guidelines of Carnegie Mellon University under which the training data was collected.


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, joblib
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

os.makedirs('models', exist_ok=True)
print('Libraries loaded ✓')

## 1. Load & Combine Datasets

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
# Update these if running locally
DESKTOP_PATH = 'Dyt-desktop.csv'
TABLET_PATH  = 'Dyt-tablet.csv'

desktop = pd.read_csv(DESKTOP_PATH, sep=';')
tablet  = pd.read_csv(TABLET_PATH,  sep=';')

desktop['source'] = 'desktop'
tablet['source']  = 'tablet'

df = pd.concat([desktop, tablet], ignore_index=True)

# Replace 'NULL' strings with NaN
df.replace('NULL', np.nan, inplace=True)

# Encode target
df['label'] = (df['Dyslexia'] == 'Yes').astype(int)

print(f'Combined dataset: {df.shape}')
print(df['label'].value_counts().rename({0:'No Dyslexia', 1:'Dyslexia'}))

In [ ]:
# Age distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Age'].hist(bins=11, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Age Distribution (combined)')
axes[0].set_xlabel('Age')

df.groupby('Age')['label'].mean().plot(kind='bar', ax=axes[1], color='salmon')
axes[1].set_title('Dyslexia Rate by Age')
axes[1].set_ylabel('Proportion')
plt.tight_layout()
plt.savefig('age_distribution.png', dpi=120)
plt.show()

## 2. Age-Group Splits & Question Sets

Following the paper's customized tablet test (Section "Customized test with new participants"):

| Group | Ages | Questions Used | Features |
|-------|------|----------------|----------|
| G1 | 7–8  | Q1–Q12, Q14–Q17, Q22–Q23, Q30 | 19 questions / 118 features |
| G2 | 9–11 | Q1–Q20, Q22–Q24, Q26–Q28, Q30 | 27 questions / 166 features |
| G3 | 12–17 | Q1–Q28, Q30–Q32 | 31 questions / 190 features |

In [ ]:
# ── Define which questions belong to each age group ───────────────────────────
AGE_GROUPS = {
    'G1_7_8':   list(range(1,13)) + list(range(14,18)) + [22, 23, 30],
    'G2_9_11':  list(range(1,21)) + [22, 23, 24] + list(range(26,29)) + [30],
    'G3_12_17': list(range(1,29)) + [30, 31, 32],
}

AGE_FILTERS = {
    'G1_7_8':   (df['Age'] >= 7)  & (df['Age'] <= 8),
    'G2_9_11':  (df['Age'] >= 9)  & (df['Age'] <= 11),
    'G3_12_17': (df['Age'] >= 12) & (df['Age'] <= 17),
}

MEASURES = ['Clicks', 'Hits', 'Misses', 'Score', 'Accuracy', 'Missrate']

def get_feature_cols(q_list):
    """Return all feature column names for a given list of question numbers."""
    cols = ['Gender', 'Nativelang', 'Otherlang', 'Age']
    for q in q_list:
        for m in MEASURES:
            cols.append(f'{m}{q}')
    return cols

for g, qs in AGE_GROUPS.items():
    n = AGE_FILTERS[g].sum()
    dys = df.loc[AGE_FILTERS[g], 'label'].sum()
    print(f'{g}: {n} participants, {dys} dyslexia ({dys/n*100:.1f}%), {len(get_feature_cols(qs))} features')

## 3. Preprocessing

In [ ]:
le_gender = LabelEncoder()
le_native = LabelEncoder()
le_other  = LabelEncoder()

df['Gender']     = le_gender.fit_transform(df['Gender'].fillna('Unknown'))
df['Nativelang'] = le_native.fit_transform(df['Nativelang'].fillna('Unknown'))
df['Otherlang']  = le_other.fit_transform(df['Otherlang'].fillna('Unknown'))

# Convert all feature columns to numeric
all_q_cols = [f'{m}{q}' for q in range(1,33) for m in MEASURES]
df[all_q_cols] = df[all_q_cols].apply(pd.to_numeric, errors='coerce')

print('Preprocessing done ✓')

## 4. Class Imbalance Strategy

The dataset is imbalanced (~10% dyslexia). We use **two complementary strategies** as recommended in recent Kaggle dyslexia notebooks:

1. **SMOTE** (Synthetic Minority Oversampling) — generates synthetic dyslexia samples in the training fold
2. **`class_weight='balanced'`** in Random Forest — downweights the majority class automatically

Both are applied inside the cross-validation loop to prevent data leakage.

In [ ]:
def build_pipeline(n_estimators=200, threshold=0.24):
    """
    Returns an imbalanced-learn Pipeline:
    Impute missing → SMOTE → Random Forest (class_weight balanced)
    
    threshold: the voting threshold from the paper (lower = higher recall for dyslexia)
    """
    pipe = ImbPipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('smote',   SMOTE(random_state=42, k_neighbors=5)),
        ('rf',      RandomForestClassifier(
                        n_estimators=n_estimators,
                        class_weight='balanced',
                        max_features='sqrt',
                        random_state=42,
                        n_jobs=-1
                    ))
    ])
    pipe.threshold = threshold
    return pipe

print('Pipeline builder ready ✓')

## 5. Train & Evaluate — One Model per Age Group

In [ ]:
# Thresholds from the paper (Table 8 for tablet groups)
THRESHOLDS = {
    'G1_7_8':   0.255,
    'G2_9_11':  0.230,
    'G3_12_17': 0.155,
}

results = {}
trained_models = {}

for group_name, q_list in AGE_GROUPS.items():
    print(f'\n{'='*55}')
    print(f'  Group: {group_name}')
    print(f'{'='*55}')
    
    mask     = AGE_FILTERS[group_name]
    feat_cols = get_feature_cols(q_list)
    # Only keep columns that actually exist in df
    feat_cols = [c for c in feat_cols if c in df.columns]
    
    X = df.loc[mask, feat_cols].values.astype(float)
    y = df.loc[mask, 'label'].values
    
    thresh = THRESHOLDS[group_name]
    pipe   = build_pipeline(threshold=thresh)
    
    # 10-fold stratified CV — get out-of-fold probabilities
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    y_proba = cross_val_predict(pipe, X, y, cv=cv, method='predict_proba')[:, 1]
    y_pred  = (y_proba >= thresh).astype(int)
    
    roc = roc_auc_score(y, y_proba)
    print(f'ROC-AUC: {roc:.3f}  |  Threshold: {thresh}')
    print(classification_report(y, y_pred, target_names=['No Dyslexia','Dyslexia']))
    
    # Confusion matrix plot
    cm = confusion_matrix(y, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Dys','Dys'], yticklabels=['No Dys','Dys'], ax=ax)
    ax.set_title(f'{group_name} — Confusion Matrix')
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'cm_{group_name}.png', dpi=120)
    plt.show()
    
    # Train FINAL model on ALL data in this group
    pipe.fit(X, y)
    trained_models[group_name] = {
        'pipeline':   pipe,
        'threshold':  thresh,
        'feat_cols':  feat_cols,
        'q_list':     q_list,
        'roc':        roc,
    }
    
    # Save to disk
    joblib.dump(trained_models[group_name], f'models/model_{group_name}.pkl')
    print(f'Saved → models/model_{group_name}.pkl')
    results[group_name] = {'roc': roc, 'n': mask.sum(), 'thresh': thresh}

## 6. Feature Importance — Top Questions per Group

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (group_name, m) in zip(axes, trained_models.items()):
    rf         = m['pipeline'].named_steps['rf']
    feat_names = m['feat_cols']
    importances = pd.Series(rf.feature_importances_, index=feat_names)
    
    # Aggregate by question number
    q_imp = {}
    for col, imp in importances.items():
        # extract question number from column name
        q_num = ''.join(filter(str.isdigit, col))
        if q_num:
            q_imp[f'Q{q_num}'] = q_imp.get(f'Q{q_num}', 0) + imp
        else:
            q_imp['Demog'] = q_imp.get('Demog', 0) + imp
    
    top = pd.Series(q_imp).nlargest(15)
    top.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'{group_name}\nTop Question Importance')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120)
plt.show()

## 7. Summary Results Table

In [ ]:
summary = pd.DataFrame(results).T
print('\n=== Final Model Summary ===')
print(summary.to_string())

# Compare against paper's Table 8 results
paper_results = pd.DataFrame({
    'Paper_ROC': {'G1_7_8': 0.663, 'G2_9_11': 0.818, 'G3_12_17': 0.806},
    'Paper_Recall_Dys': {'G1_7_8': 72.2, 'G2_9_11': 75.8, 'G3_12_17': 78.1},
})
comparison = summary.join(paper_results)
print('\n=== Comparison with Paper (Table 8) ===')
print(comparison.to_string())

## 8. How to Use the Models in Lexora (Next.js / Python API)

In [ ]:
# ── Inference helper ─────────────────────────────────────────────────────────
# This is the function you call from your FastAPI / Flask endpoint

import joblib
import numpy as np

def load_models():
    """Load all three age-group models."""
    return {
        'G1_7_8':   joblib.load('models/model_G1_7_8.pkl'),
        'G2_9_11':  joblib.load('models/model_G2_9_11.pkl'),
        'G3_12_17': joblib.load('models/model_G3_12_17.pkl'),
    }

def get_group(age: int) -> str:
    """Map participant age → model group name."""
    if 7 <= age <= 8:   return 'G1_7_8'
    if 9 <= age <= 11:  return 'G2_9_11'
    if 12 <= age <= 17: return 'G3_12_17'
    raise ValueError(f'Age {age} out of supported range (7–17)')

def predict_dyslexia(age: int, feature_dict: dict, models: dict) -> dict:
    """
    Predict dyslexia risk for one participant.
    
    Parameters
    ----------
    age          : int, participant age (7–17)
    feature_dict : dict mapping column names → values
                   e.g. {'Gender': 0, 'Age': 9, 'Clicks1': 5, 'Hits1': 4, ...}
    models       : dict returned by load_models()
    
    Returns
    -------
    dict with keys: group, probability, risk_level, recommendation
    """
    group  = get_group(age)
    m      = models[group]
    pipe   = m['pipeline']
    thresh = m['threshold']
    cols   = m['feat_cols']
    
    # Build feature vector — missing questions get NaN (imputer handles them)
    X = np.array([[feature_dict.get(c, np.nan) for c in cols]], dtype=float)
    
    proba      = pipe.predict_proba(X)[0, 1]
    prediction = int(proba >= thresh)
    
    if proba < 0.30:
        risk = 'Low'
        rec  = 'No further screening indicated at this time.'
    elif proba < 0.60:
        risk = 'Moderate'
        rec  = 'Consider follow-up assessment with a specialist.'
    else:
        risk = 'High'
        rec  = 'Professional evaluation is strongly recommended.'
    
    return {
        'group':          group,
        'probability':    round(float(proba), 4),
        'risk_level':     risk,
        'recommendation': rec,
        'disclaimer':     (
            'This result is a screening estimate only and does NOT constitute '
            'a diagnosis. Please consult a qualified professional for a comprehensive assessment.'
        )
    }

# ── Quick smoke test ─────────────────────────────────────────────────────────
models_loaded = load_models()

# Build a fake participant: age 10, all features = 0
fake_features = {'Gender': 0, 'Nativelang': 1, 'Otherlang': 0, 'Age': 10}
for q in range(1, 33):
    for m in ['Clicks','Hits','Misses','Score','Accuracy','Missrate']:
        fake_features[f'{m}{q}'] = 0.0

result = predict_dyslexia(age=10, feature_dict=fake_features, models=models_loaded)
print('Smoke test result:')
for k, v in result.items():
    print(f'  {k}: {v}')

## 9. Questions by Age Group — Web Integration Guide

This section maps which questions to **show or hide** on the frontend based on participant age.
Your Next.js app should:
1. Collect `age` during onboarding
2. Call `/api/get-questions?age=N` → returns which Q numbers to render
3. After test completion, POST results to `/api/predict` with the feature dict

In [ ]:
import json

# This JSON config is what your Next.js API reads
question_config = {
    'groups': {
        'G1_7_8': {
            'age_range': [7, 8],
            'questions': AGE_GROUPS['G1_7_8'],
            'model_file': 'model_G1_7_8.pkl',
            'threshold':  THRESHOLDS['G1_7_8'],
            'description': '7–8 year olds: phonological basics, letter recognition, working memory'
        },
        'G2_9_11': {
            'age_range': [9, 11],
            'questions': AGE_GROUPS['G2_9_11'],
            'model_file': 'model_G2_9_11.pkl',
            'threshold':  THRESHOLDS['G2_9_11'],
            'description': '9–11 year olds: full phonological + lexical + morphological battery'
        },
        'G3_12_17': {
            'age_range': [12, 17],
            'questions': AGE_GROUPS['G3_12_17'],
            'model_file': 'model_G3_12_17.pkl',
            'threshold':  THRESHOLDS['G3_12_17'],
            'description': '12–17 year olds: complete battery including dictation + working memory'
        }
    },
    'measures_per_question': ['Clicks', 'Hits', 'Misses', 'Score', 'Accuracy', 'Missrate'],
    'demographic_features':  ['Gender', 'Nativelang', 'Otherlang', 'Age']
}

with open('question_config.json', 'w') as f:
    json.dump(question_config, f, indent=2)

print(json.dumps(question_config, indent=2))

---
## Summary

| Group | Ages | Participants | Questions | ROC (paper) |
|-------|------|-------------|-----------|-------------|
| G1 | 7–8 | ~911 desktop + ~327 tablet | 19 | 0.663–0.782 |
| G2 | 9–11 | ~1,628 desktop + ~567 tablet | 27 | 0.818–0.878 |
| G3 | 12–17 | ~1,105 desktop + ~498 tablet | 31 | 0.806–0.851 |

**Files produced:**
- `models/model_G1_7_8.pkl`
- `models/model_G2_9_11.pkl`  
- `models/model_G3_12_17.pkl`
- `question_config.json` — consumed by the Next.js API

**Reference:** Rello, L. et al. (2020). Predicting risk of dyslexia with an online gamified test. *PLOS ONE*, 15(12), e0241687.